In [0]:
import sys
from pyspark.sql.functions import col, concat_ws
from pyspark.sql.types import (
    BooleanType, IntegerType, LongType, DoubleType, DecimalType, 
    DateType, TimestampType, StringType
)

parent_path = '/Workspace/Users/esymphony.life@gmail.com/MSF_test/customer'
if parent_path not in sys.path:
    sys.path.insert(0, parent_path)

from common.functions import recursive_explode

In [0]:
def infer_datatypes(df):
    """
    Returns a DataFrame with original data but updated datatypes using best fit inference.
    """
    priority_types = [
        BooleanType(),
        IntegerType(),
        LongType(),
        DoubleType(),
        DecimalType(38, 18),
        DateType(),
        TimestampType()
    ]
    
    best_types = {}
    for field in df.schema.fields:
        col_name = field.name
        orig_type = field.dataType
        # Only try to infer for StringType columns
        if not isinstance(orig_type, StringType):
            best_types[col_name] = orig_type
            continue
        for t in priority_types:
            test_df = df.select(
                col(col_name).alias("orig"),
                col(col_name).cast(t).alias("cast_val")
            )
            failed_cast_count = test_df.filter(
                col("orig").isNotNull() & col("cast_val").isNull()
            ).count()
            if failed_cast_count == 0:
                best_types[col_name] = t
                break
        else:
            best_types[col_name] = StringType()
    
    # Cast columns to inferred types
    for col_name, dtype in best_types.items():
        df = df.withColumn(col_name, col(col_name).cast(dtype))
    return df

In [0]:
def write_silver(catalog, schema, domain):
    """
    Orchestrates the Bronze-to-Silver movement with flattening and typing.
    """
    # 1. Fetch tables from Information Schema
    tables_df = spark.read.table(f"{catalog}.information_schema.tables")\
                    .where(col("table_schema") == domain)\
                    .select(concat_ws(".", col("table_catalog"), col("table_schema"), col("table_name")).alias("full_table_path"))
    
    table_paths = [row.full_table_path for row in tables_df.collect()]

    for table in table_paths:
        write_path = table.replace("bronze", "silver")
        
        print(f"\nProcessing {table}...")
        
        # Load raw Bronze
        raw_df = spark.read.table(table)
        
        # Process: Explode -> Type Correct -> Audit Columns
        silver_df = recursive_explode(raw_df)
        print(f"Flattened to {len(silver_df.columns)} columns")
        
        silver_df = infer_datatypes(silver_df)
        
        silver_df = silver_df.withColumn("process_dts", current_timestamp())

        # Write to Silver Layer
        silver_df.write \
            .format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable(write_path)

        print(f"Successfully processed Silver table: {write_path}")

In [0]:
write_silver(dbutils.widgets.get("CATALOG"), dbutils.widgets.get("SCHEMA"), dbutils.widgets.get("DOMAIN"))